# TF-C study, part 2: figures of the ablation stages on FordA

Figures of the ablation runs (D96, D97) listed in `configs/plots_forda.yaml`: loss, collapse measures, spectra,
probes and PCA. The plotting functions are in `plots.py`; each cell below draws one figure. Nothing is written to
disk: the figures exist only in this notebook. **Clear all outputs before committing** (VS Code: "Clear All
Outputs"; Jupyter: Edit → Clear Outputs of All Cells), as for the notebooks in `eda/` (D68).

**Run:** `uv sync --group eda`, then open this notebook (VS Code with the repo's `.venv`, or
`uv run --group eda jupyter lab`) and run all cells. Add D and E to the config once they have run.

**Reading the figures.** Only measured values are drawn. With several seeds a line is the mean over seeds and the
band runs from the smallest to the largest seed. Colours: A and B are both blue ("one branch alone"; they never
share a panel, A has only h_T and B only h_F), C aqua, D yellow, E (full TF-C) black; every line also carries its
stage letter. Panels are always in the same place: top row h_T, h_F, h; bottom row z_T, z_F, z. z exists only in
D and E (in A–C no loss reaches the projectors).

## Setup

In [ ]:
import os
import sys
from pathlib import Path


def repo_root():
    """The repository root: the first folder upwards that contains pyproject.toml."""
    p = Path.cwd().resolve()
    while not (p / "pyproject.toml").exists():
        p = p.parent
    return p


ROOT = repo_root()
os.chdir(ROOT)  # run folders and data paths in the configs are relative to the repo root
sys.path.insert(0, str(ROOT))

from studies.tfc_ablation import plots

CONFIG = "studies/tfc_ablation/configs/plots_forda.yaml"
cfg, runs = plots.load(CONFIG)
for st, r in runs.items():
    print(f"stage {st}: {r['folder']}, seeds {r['seeds']}")

## 1. Loss
Each loss part per epoch: means over batches, unweighted (in D and E the loss uses λ times L_T and L_F).
Dotted lines: ln(2B − 1) = ln 127, the NT-Xent value when all candidates look alike; for L_C, 3δ, its value
when S_TF = S_pair. In A and B only L_T or L_F exists; S_TF and L_C are logged (not trained) in C.

In [ ]:
plots.loss_figure(runs)

## 2. Dimensionality
Effective rank (RankMe), the number of directions that hold the chosen share of the variance (from
`spectra.csv`), and the fraction of dead dimensions. Dimensional collapse: windows still differ, but the
variance sits in few directions (low rank, few directions). "Dead" is relative to the largest std of that
representation (D94), so it also rises when dimensions shrink unevenly without collapsing.

In [ ]:
plots.measure_figure(runs, "rankme", "Effective rank (RankMe)")

In [ ]:
share = f"{cfg['variance_share']:.0%}"
plots.measure_figure(runs, "dirs", f"Directions holding {share} of variance",
                     note=f"Smallest number of directions (largest eigenvalues first) that hold {share} of the variance.")

In [ ]:
plots.measure_figure(runs, "frac_dead", "Fraction of dead dimensions",
                     note="Dead = std below 1 % of the largest std of that representation (D94).")

## 3. Variance distribution
First: the cumulative share of variance in the top k directions at the last epoch, stages overlaid (a curve that
rises steeply = variance in few directions). Then per stage: the eigenvalue spectrum (divided by the total) and the
cumulative share at the epochs in `spectrum_epochs`, light = early, dark = late; first seed.

In [ ]:
plots.final_spectrum_figure(runs)

In [ ]:
for st, r in runs.items():
    plots.spectrum_figure(st, r, cfg["spectrum_epochs"])

## 4. Scale
Mean norm and mean per-dimension std (log axis). Kept apart from shape: in A and B the scale shrinks while rank
and spread stay healthy.

In [ ]:
plots.measure_figure(runs, "mean_norm", "Mean norm", log=True)

In [ ]:
plots.measure_figure(runs, "mean_std", "Mean per-dimension std", log=True)

## 5. Geometry
Mean cosine between different windows (1 = all windows point the same way, complete collapse), the cosine of each
window with its augmented view, and the two against each other per epoch (○ epoch 0, ■ last epoch). Good: low
cosine between windows, high alignment with the augmented view.

In [ ]:
plots.measure_figure(runs, "mean_cosine", "Mean cosine between windows")

In [ ]:
plots.measure_figure(runs, "alignment", "Cosine with augmented view")

In [ ]:
plots.trajectory_figure(runs)

## 6. SIGReg statistic
Epps–Pulley test of 1-D projections against N(0, 1) (official LeJEPA settings, D94). Not scale-free: a shrinking
scale raises it without any change of shape, so read it together with section 4.

In [ ]:
plots.measure_figure(runs, "sigreg", "SIGReg statistic", log=True,
                     note="Not scale-free: shrinking scale raises it without any change of shape. 0 = standard Gaussian.")

## 7. Probes
FordA test accuracy against the label budget (D26 probes, D40 budgets), per probe and representation. Grey:
TS2Vec (dashed) and T-Loss (dotted) with the same probe, mean over their seeds. Runs without `probes.csv` yet
are left out (a note is printed when loading).

In [ ]:
plots.probe_figure(runs, cfg["references"])

## 8. PCA
First two principal components of the diagnostic windows at epoch 0 and the last epoch, coloured by class
(labels used for colour only). Epoch 0 is rebuilt from the seed; both models are checked against the stored
spectra before drawing. Needs the FordA data and `pretrained.pt`; about a minute on the CPU.

In [ ]:
for st, r in runs.items():
    plots.pca_figure(st, r, cfg["pca_device"])